# Session 8: pandas II, cleaning the mess (Project milestone 1)

**KLAi | AI and Data Science Professional**

You clean a raw export of Lakeside Mart order lines (fictional, generated data) with eight planted defects. After each cleaning step you add a row to the data cleaning log. At the end you reconcile your cleaned total to the control total and name the reconciliation difference. Each step has the same rhythm: **PREDICT**, run, **READ**, then **CHANGE one thing**. A cell that says `None  # your code here` is yours to fill. The check cells say OK or NOT YET, and they keep running while an answer is still `None`.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
# Helper for the checks. Run this cell once. You do not need to read it.
# attempt() runs a check on your answer. If the answer is not written yet it says NOT YET instead of stopping with an error.
def attempt(label, expr, expected, tol=0.01):
    try:
        value = eval(expr)
        want = eval(expected) if isinstance(expected, str) else expected
    except Exception:
        value, want = None, 0
    check(label, value, want, tol)

---
## Step 1: Load the raw file, copy it and set up the log

This first cell is given, so every later step has a `raw` and a `df` to work on. We never edit the raw file. We clean a copy called `df`. The helper `note` adds one row to the data cleaning log: the step, the rows before, the rows after and the reason.

**PREDICT first.** The raw export has one record for each product on an order. How many rows do you expect: hundreds, thousands or tens of thousands?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
import pandas as pd

raw = pd.read_csv('orders_raw.csv')   # given: the raw file, never changed
df = raw.copy()                       # given: the copy that you clean

log = []
def note(step, before, after, why):
    log.append({"step": step, "rows_before": before, "rows_after": after, "why": why})

print(raw.shape)
print(raw.dtypes)
check("raw rows", raw.shape[0], 41259, 0)
check("df is a copy of raw", len(df), len(raw), 0)

**READ the output.** Which columns are stored as object? Which of them should be a number or a date?

> My answer: 

---
## Step 2: Duplicates

An exact duplicate repeats another record in every variable. drop_duplicates keeps the first copy. line_id is the primary key, so it must be unique afterwards.

**PREDICT first.** Roughly how many exact duplicate lines do you expect in 41 thousand lines: none, tens or thousands?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: remember len(df) in before; df.duplicated().sum() counts the repeats; then df.drop_duplicates()
before = len(df)
n_dupes = None  # your code here: the number of duplicate lines
df = df  # your code here: drop the duplicates
note("Drop exact duplicates", before, len(df), str(n_dupes) + " identical lines")

In [ ]:
try:
    print(n_dupes, len(df), df["line_id"].is_unique)
except Exception:
    print("Not yet: finish the cell above, then run this cell again.")
attempt('duplicates found', 'n_dupes', 80, 0)
attempt('rows after', 'len(df)', 41179, 0)
attempt('line_id is unique', 'df["line_id"].is_unique', 1, 0)

**CHANGE one thing.** Run df.duplicated(subset=["line_id"]).sum() before dropping (use raw). Is it the same number? Why?

> What I saw: 

---
## Step 3: Data types: prices and dates

to_numeric converts text to numbers, so remove the text UGX and the comma first. to_datetime reads dates: give it a format and errors="coerce", read each style on its own, then fillna joins the two. Dates with slashes are day/month/year.

**PREDICT first.** After to_numeric, what data type will unit_price have? After the two readings, how many dates should still be missing?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: str.replace('UGX', '') then str.replace(',', '') then pd.to_numeric. Formats: '%Y-%m-%d' and '%d/%m/%Y'
p = None  # your code here: unit_price with UGX and the comma removed
df["unit_price"] = None  # your code here: convert p to numbers
iso = None  # your code here: order_date read as year-month-day, errors="coerce"
uk = None   # your code here: order_date read as day/month/year, errors="coerce"
df["order_date"] = None  # your code here: iso, with the gaps filled from uk

In [ ]:
note("Price text to number", len(df), len(df), "UGX and commas removed, to_numeric")
note("Two date formats to dates", len(df), len(df), "slashes read day first")
try:
    print(df["unit_price"].dtype, df["order_date"].isna().sum(), df["order_date"].min().date(), df["order_date"].max().date())
except Exception:
    print("Not yet: finish the cell above, then run this cell again.")
attempt('prices are numbers', 'df["unit_price"].dtype.kind in "iuf"', 1, 0)
attempt('dates missing', 'df["order_date"].isna().sum()', 0, 0)
attempt('slash dates', 'iso.isna().sum()', 12383, 0)
attempt('latest date is 28 Dec 2025', 'df["order_date"].max() == pd.Timestamp("2025-12-28")', 1, 0)

**READ the output.** iso is missing for the slash dates and uk for the dash dates. Why do the two numbers add up to more than the number of rows?

> My answer: 

**CHANGE one thing.** Read the dates month first, format "%m/%d/%Y", into a new Series called us. How many become NaT? Do not store it in df.

> What I saw: 

---
## Step 4: Text: branch and payment

String methods fix text: strip, lower, title, capitalize and replace with a dictionary. Do strip and lower or title first, then the dictionary.

**PREDICT first.** The raw branch has 26 spellings and payment has 11. After cleaning, how many do you expect of each?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: branch: str.strip().str.title() then replace({'Kampala Cntral': 'Kampala Central', 'Entebe': 'Entebbe'}). payment: str.strip().str.lower(), replace({'mm': 'mobile money'}), str.capitalize()
df["branch"] = None   # your code here
df["payment"] = None  # your code here

In [ ]:
note("Branch and payment text", len(df), len(df), "26 branch spellings to 8, 11 payment spellings to 3")
try:
    print(df["payment"].value_counts())
except Exception:
    print("Not yet: finish the cell above, then run this cell again.")
attempt('branches', 'df["branch"].nunique()', 8, 0)
attempt('payment methods', 'df["payment"].nunique()', 3, 0)
attempt('Mobile money lines', 'df["payment"].value_counts()["Mobile money"]', 19689, 0)

**READ the output.** Which spelling had a trailing space in the raw file? Which string method removed it?

> My answer: 

---
## Step 5: Missing values: flag, fill and drop

Flag first (an indicator variable), then fill the blank customers with the constant walk-in. The 12 lines with a blank qty cannot be recovered, so drop only those, with subset.

**PREDICT first.** What would happen to the row count if you called df.dropna() on the whole table instead of using subset? Write a guess, then try it in a new cell, without storing it.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: df['customer_id'].isna() gives the flag; fillna('walk-in'); dropna(subset=['qty'])
df["customer_missing"] = None  # your code here: True where customer_id is missing
df["customer_id"] = None  # your code here: fill the blanks with "walk-in"
lost = None  # your code here: the lines with a blank qty
df = df  # your code here: drop those lines

In [ ]:
try:
    note("Flag and fill blank customers", len(df) + len(lost), len(df) + len(lost), "blanks flagged, then labelled walk-in")
    note("Drop lines with blank qty", len(df) + len(lost), len(df), str(len(lost)) + " lines, quantity cannot be recovered")
    print(df["customer_missing"].sum(), (df["customer_id"] == "walk-in").sum(), len(lost), len(df))
except Exception:
    print("Not yet: finish the cell above, then run this cell again.")
attempt('customers flagged', 'df["customer_missing"].sum()', 10626, 0)
attempt('walk-in lines', '(df["customer_id"] == "walk-in").sum()', 12456, 0)
attempt('lines dropped', 'len(lost)', 12, 0)
attempt('rows left', 'len(df)', 41167, 0)

**READ the output.** Why is the walk-in count larger than the flag count?

> My answer: 

**CHANGE one thing.** In a new cell, run len(df.dropna()) and compare it with len(df). Why is it so much smaller? Delete the cell afterwards.

> What I saw: 

---
## Step 6: Outliers: check before you remove

Returns are lines with a negative qty. The IQR rule flags a value beyond 1.5 times the interquartile range from the quartiles. A flag is a question, so check each kind of flag and write down what you decide.

**PREDICT first.** Will any unit_price differ from the price list in products.csv? Will the IQR rule flag any revenue values?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: merge on product_id, compare unit_price with unit_price_ugx. revenue = qty * unit_price * (1 - discount_pct / 100). quantile(0.25) and quantile(0.75)
prod = pd.read_csv("products.csv")
n_mismatch = None  # your code here: lines whose unit_price differs from the price list
n_returns = None   # your code here: lines with a negative qty
df["revenue"] = None  # your code here: qty * unit_price * (1 - discount_pct / 100)
q1 = None  # your code here
q3 = None  # your code here
upper_fence = None  # your code here: q3 + 1.5 * (q3 - q1)
n_flagged = None  # your code here: revenue values above the upper fence
lower_fence = None  # your code here: q1 - 1.5 * (q3 - q1)
n_flagged_low = None  # your code here: revenue values below the lower fence

In [ ]:
try:
    note("Check outliers, keep returns", len(df), len(df), str(n_returns) + " returns kept. IQR rule on revenue: " + str(n_flagged) + " high and " + str(n_flagged_low) + " low lines, all kept")
    print(n_mismatch, n_returns, round(upper_fence), n_flagged, n_flagged_low)
except Exception:
    print("Not yet: finish the cell above, then run this cell again.")
attempt('price mismatches', 'n_mismatch', 0, 0)
attempt('returns', 'n_returns', 1179, 0)
attempt('upper fence', 'upper_fence', 29275.0, 0.5)
attempt('lines above the fence', 'n_flagged', 2130, 0)
attempt('lines below the lower fence', 'n_flagged_low', 385, 0)

**READ the output.** Are the flagged lines errors? The low flags are mostly returns: why? What would you check before removing a flagged line?

> My answer: 

---
## Step 7: Reconcile to the control total

The till system reports a control total of 561,421,125 UGX for the same lines. Your cleaned total is the sum of revenue. The reconciliation difference is the control total minus the cleaned total. Do not call it a residual: that word belongs to models.

**PREDICT first.** Do you expect the reconciliation difference to be zero? What could make it differ?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: cleaned_total = round(df["revenue"].sum()); difference = control_total - cleaned_total
control_total = 561421125
cleaned_total = None  # your code here: the sum of revenue, rounded to a whole number
reconciliation_difference = None  # your code here

In [ ]:
try:
    lost_unit_value = (lost["unit_price"] * (1 - lost["discount_pct"] / 100)).sum()
    units_each = reconciliation_difference / lost_unit_value
    print(cleaned_total, reconciliation_difference, round(units_each, 1))
except Exception:
    print("Not yet: finish the cell above, then run this cell again.")
attempt('cleaned total', 'cleaned_total', 561288435, 0)
attempt('reconciliation difference', 'reconciliation_difference', 132690, 0)
attempt('units per dropped line', 'units_each', 1.978, 0.01)

**READ the output.** The difference divided by the value of one unit of each dropped line is about 2. What does that suggest about the cause? Why is it an estimate and not proof?

> My answer: 

**Your reconciliation note (Project milestone 1).** In one or two sentences, in your own words, say what the reconciliation difference is and what most likely causes it.

> My note: 

---
## Step 8: Read the log and save the cleaned file

The log is the audit trail. It should have seven rows and the last row should end at the number of rows you keep. The next cell saves the cleaned data as orders_clean.csv and the log as cleaning_log.csv with to_csv.

In [ ]:
# Hint: pd.DataFrame(log) turns the list into a table. df.to_csv(name, index=False) saves the file
log_table = None  # your code here: the log as a DataFrame, from pd.DataFrame(log)

In [ ]:
try:
    df.to_csv("orders_clean.csv", index=False)
    log_table.to_csv("cleaning_log.csv", index=False)
    print(log_table.to_string())
except Exception:
    print("Not yet: finish the cell above, then run this cell again.")
attempt('log rows', 'len(log_table)', 7, 0)
attempt('last log row ends at', 'log_table["rows_after"].iloc[-1]', 41167, 0)

**CHANGE one thing.** Add a one-line manager summary as a comment in the cell above: what changed, and what the reconciliation difference is.

> What I saw: 

---
## Finish

Run the cell below. It should say **All 26 checks pass**. Then download `orders_clean.csv` and `cleaning_log.csv`, save the notebook to your Drive, and keep the notebook with your log and your reconciliation note. Show them to your tutor for feedback. Nothing here is marked.

In [ ]:
show_checks()